In [2]:
from typing import List

import torch
from at2v.dloader import TagDataset
from at2v.shared import get_setup
from torch.utils.data import DataLoader
import torch.nn.functional as F

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data, tagtok, anitag2vec = get_setup(
    device=device,
    prefix_path= ".."
)

anitag2vec.load_state_dict(torch.load("../checkpoints/anitag2vec_e15_s25000_p905216.pth"))
anitag2vec.to(device)
anitag2vec.eval()

Loading tokenizer from '../checkpoints/token_vocab_size_5000_freq_3.json'..
Done!


AniTag2Vec(
  (emb): Embedding(5100, 64)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (linproj): Linear(in_features=128, out_features=128, bias=True)
)

In [4]:
def to_dataloader(inputs: List[List[str]]):
    dataset = TagDataset(
        list_of_tags=inputs,
        max_len_cut=64,
        tokenizer=tagtok
    )
    return DataLoader(dataset, batch_size=len(inputs), shuffle=False)

def run_inference(inputs: List[List[str]]):
    batches = to_dataloader(inputs)
    for batch in batches:
        batch = batch.to(device)
        return anitag2vec(batch)

def rank_cosim(query: List[str], items: List[List[str]]):
    q = F.normalize(run_inference([query]), dim=1)   # (1, O)
    xs = F.normalize(run_inference(items), dim=1)    # (N, O)
    scores = (q @ xs.T).squeeze(0)                   # (N,)

    indices = torch.argsort(scores, descending=True)
    ranked_items = [items[i] for i in indices.tolist()]

    return list(zip(scores[indices], ranked_items))

In [5]:
query = data.real_examples[600]
ranked = rank_cosim(
    query,
    data.real_examples[2000:2500]
)

In [ ]:
print("Closest to:", ", ".join(query))
for index, (cosim, tags) in enumerate(ranked[:3]):
    perc = (1 + cosim.item()) / 2
    print(f"# {index + 1}, {round(perc * 100 * 100) / 100}%: {', '.join(sorted(tags))}")

Closest to: yojouhan, english, translated, shijou sadafumi, sole female, tanlines, tomboy, sole male, bokutachi wa benkyou ga dekinai, doujinshi, glasses, schoolgirl uniform
# 1, 91.43%: aoi manabu, big breasts, bluemage, bokutachi wa benkyou ga dekinai, cosplaying, doujinshi, english, facesitting, gloves, mafuyu kirisu, maid, mind control, nakadashi, nariyuki yuiga, paizuri, sole male, stockings, teacher, tracksuit, translated
# 2, 87.35%: bbm, doku doku kinoko, doujinshi, emotionless sex, glasses, japanese, jujutsu kaisen, maki zenin, mind control, mosaic censorship, nakadashi, ponytail, rape, schoolgirl uniform, sex toys, shinonome 108, sole female
# 3, 83.88%: amagi brilliant park, anal, big breasts, dilf, double penetration, doujinshi, eitarou, english, full censorship, isuzu sento, kaminari neko, lolicon, low lolicon, military, mind break, netorare, prostitution, sex toys, stockings, translated
# 4, 83.3%: ahegao, big breasts, bodystocking, bodysuit, doujinshi, english, f.w.zholi